In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib qt

In [ ]:
import joblib
import numpy as np
import scipy as scp
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

from matplotlib.colors import LinearSegmentedColormap, LogNorm, PowerNorm
from matplotlib.lines import Line2D
from scipy.signal import windows
from scipy.stats import norm
from utils import dm_test
from tqdm import tqdm

In [ ]:
mpl.rcParams.update({
    # Requires a LaTeX install (TeX Live / MiKTeX). Without one, set
    # 'text.usetex': False and 'mathtext.fontset': 'cm' instead.
    "pgf.texsystem"       : "xelatex",
    'text.usetex'         : False,
    'text.latex.preamble' : r'\usepackage{amsmath}',
    'font.family'         : 'serif',   # Computer Modern = default LaTeX font

    # Match your document's font sizes (most journals: 10 pt)
    'font.size'             : 10,
    'axes.labelsize'        : 10,
    'legend.title_fontsize' : 9.2,
    'xtick.labelsize'       : 9,
    'ytick.labelsize'       : 9,
    'legend.fontsize'       : 9,

    # Okabe–Ito palette — colorblind-safe, one line to replace the default cycle
    'axes.prop_cycle': mpl.cycler('color', [
        '#0072B2', '#56B4E9', '#D55E00', '#E69F00', 
        '#009E73', '#F0E442', '#CC79A7', '#000000'
    ]),

    'lines.linewidth'  : 1.5,
    'axes.linewidth'   : 0.8,
})

okabe_ito = [
        '#0072B2', '#56B4E9', '#D55E00', '#E69F00', 
        '#009E73', '#F0E442', '#CC79A7', '#000000'
    ]

mm = 1 / 25.4

# Aux. Functions
---

In [ ]:
def bootstrap(x, n_boot=5000, random_state=None):
    rng = np.random.default_rng(random_state)
    x = np.asarray(x)   # shape (n_samples, h)
    n, h = x.shape

    boot_means = np.empty((n_boot, h))
    for i in tqdm(range(n_boot), desc='Bootstrapping'):
        idx = rng.choice(n, size=n, replace=True)
        boot_means[i] = x[idx].mean(axis=0)

    boot_mean_curve = boot_means.mean(axis=0)
    boot_std_curve = boot_means.std(axis=0, ddof=1)

    low_boot_gauss = boot_mean_curve - 1.645 * boot_std_curve # c.i. 90%
    high_boot_gauss = boot_mean_curve + 1.645 * boot_std_curve

    low_boot_pct = np.percentile(boot_means, 5, axis=0) # ignore 5% low
    high_boot_pct = np.percentile(boot_means, 95, axis=0) # ignore 5% high -> total 10% -> should be close to c.i. 90%

    return boot_mean_curve, low_boot_gauss, high_boot_gauss, low_boot_pct, high_boot_pct

In [ ]:
def calculate_full_nmse(results, amt_data, nb_samples, samples_spacing):
    y_lin = results[amt_data]['y_lin'][:, :nb_samples*samples_spacing:samples_spacing, :500] # N_exp, N_samples, h_max
    y_true = results[amt_data]['y_true'][:, :nb_samples*samples_spacing:samples_spacing, :500] # N_exp, N_samples, h_max
    y_nonlin = results[amt_data]['y_nonlin'][:, :nb_samples*samples_spacing:samples_spacing, :500] # N_exp, N_samples, h_max
    X_train = results[amt_data]['X_train']
    
    # Variance of train data
    var_val = X_train[:, 1, :].var(axis=-1)[:, np.newaxis, np.newaxis]
    
    # NMSE at horizon h (N_exp, N_samples, 500)
    inst_nmse_lin = (y_true - y_lin)**2 / var_val
    inst_nmse_nonlin = (y_true - y_nonlin)**2 / var_val
    
    # Running NMSE (N_exp, N_samples, 500)
    t_divisor = np.arange(1, 501)
    run_nmse_lin = np.cumsum(inst_nmse_lin, axis=2) / t_divisor
    run_nmse_nonlin = np.cumsum(inst_nmse_nonlin, axis=2) / t_divisor
    
    return inst_nmse_lin, inst_nmse_nonlin, run_nmse_lin, run_nmse_nonlin

In [ ]:
def spectral_distance(results, amt_data, nb_samples, samples_spacing, dt, window_size=500):

    y_lin = results[amt_data]['y_lin'][:, :nb_samples*samples_spacing:samples_spacing, :window_size] # (N_exp, N_samples, h_max)
    y_true = results[amt_data]['y_true'][:, :nb_samples*samples_spacing:samples_spacing, :window_size]
    y_nonlin = results[amt_data]['y_nonlin'][:, :nb_samples*samples_spacing:samples_spacing, :window_size]

    y_true = y_true - y_true.mean(axis=2, keepdims=True)
    y_lin = y_lin - y_lin.mean(axis=2, keepdims=True)
    y_nonlin = y_nonlin - y_nonlin.mean(axis=2, keepdims=True)

    # Hann window
    w = windows.hann(window_size)[np.newaxis, np.newaxis, :]

    # Apply window and compute FFT
    fft_y_true = np.fft.rfft(y_true * w, axis=2) # FFT in the temporal dimension: t -> f
    fft_y_lin = np.fft.rfft(y_lin * w, axis=2)
    fft_y_nonlin = np.fft.rfft(y_nonlin * w, axis=2)

    # Frequency axis
    freqs = np.fft.rfftfreq(window_size, d=dt)

    # Power spectra
    spectrum_y_true = np.abs(fft_y_true)**2
    spectrum_y_lin = np.abs(fft_y_lin)**2
    spectrum_y_nonlin = np.abs(fft_y_nonlin)**2

    norm_spectrum_y_true = spectrum_y_true / spectrum_y_true.sum(axis=2, keepdims=True) # Divide by the total power 
    norm_spectrum_y_lin = spectrum_y_lin / spectrum_y_lin.sum(axis=2, keepdims=True)
    norm_spectrum_y_nonlin = spectrum_y_nonlin / spectrum_y_nonlin.sum(axis=2, keepdims=True)

    # A. WITHOUT NORMALIZATION
    spectral_distance_lin = np.sum(np.abs(spectrum_y_lin - spectrum_y_true), axis=2) / (np.sum(spectrum_y_true, axis=2) + 1e-12) # L1 norm normalized by the ground truth
    spectral_distance_nonlin = np.sum(np.abs(spectrum_y_nonlin - spectrum_y_true), axis=2) / (np.sum(spectrum_y_true, axis=2) + 1e-12)


    # B. WITH NORMALIZATION
    norm_spectral_distance_lin = np.sum(np.abs(norm_spectrum_y_lin - norm_spectrum_y_true), axis=2) / (np.sum(norm_spectrum_y_true, axis=2) + 1e-12) # L1 norm normalized by the ground truth
    norm_spectral_distance_nonlin = np.sum(np.abs(norm_spectrum_y_nonlin - norm_spectrum_y_true), axis=2) / (np.sum(norm_spectrum_y_true, axis=2) + 1e-12)

    # Results
    spectral_dist_results = {
        'spectral_distance_lin': spectral_distance_lin,
        'spectral_distance_nonlin': spectral_distance_nonlin,

        'norm_spectral_distance_lin': norm_spectral_distance_lin,
        'norm_spectral_distance_nonlin': norm_spectral_distance_nonlin
    }

    return spectral_dist_results

# Load data
---

In [ ]:
results = joblib.load(r"data\results.joblib")

# Scaling law (extra)
---

In [ ]:
horizons = [1, 250//3, 500//3, 2*500//3, 500]
N_train_list = [1000, 2000, 3000, 4000, 5000, 7500, 10000, 12500, 15000, 20000, 25000, 30000]

nmse_lin_scaling = {h: [] for h in horizons}
nmse_nonlin_scaling = {h: [] for h in horizons}

for N_train in N_train_list:
    inst_nmse_lin, inst_nmse_nonlin, run_nmse_lin, run_nmse_nonlin = calculate_full_nmse(results=results, amt_data=N_train, nb_samples=5000, samples_spacing=1)

    mean_running_lin = run_nmse_lin.mean(axis=1).mean(axis=0) # Doesn't matter average samples inside an experiment than average experiments, or average experiment samples to than average samples
    mean_running_nonlin = run_nmse_nonlin.mean(axis=1).mean(axis=0)

    for h in horizons:
        nmse_lin_scaling[h].append(mean_running_lin[h - 1])
        nmse_nonlin_scaling[h].append(mean_running_nonlin[h - 1])

fig, ax = plt.subplots(figsize=(250 * mm, 100 * mm), layout='constrained')

cmap = plt.cm.viridis
colors = [cmap(i) for i in np.linspace(0, 0.9, len(horizons))]

for idx, h in enumerate(horizons):
    color = colors[idx]
    ax.plot(N_train_list, nmse_lin_scaling[h], marker='o', linestyle='-', color=color)
    ax.plot(N_train_list, nmse_nonlin_scaling[h], marker='s', linestyle='--', color=color)

ax.set_xlabel(r"Amount of train data ($N_{train}$)")
ax.set_ylabel(r"NMSE")
ax.set_title(r"Scaling Law: Linear vs Nonlinear Optical Features")
ax.set_xscale('log')
ax.set_yscale('log')
ax.spines[['top', 'right']].set_visible(False)
ax.minorticks_on()
ax.grid(which='major', linestyle='-', alpha=0.2)
ax.grid(which='minor', linestyle='--', alpha=0.1)


# Opt. features legend
feature_handles = [
    Line2D([0], [0], color=okabe_ito[-1], linestyle='-', marker='o', label='Linear'),
    Line2D([0], [0], color=okabe_ito[-1], linestyle='--', marker='s', label='Nonlinear')
]
leg_features = ax.legend(handles=feature_handles, title='Optical Features', 
                         bbox_to_anchor=(1.01, 0.55), loc='upper left', frameon=True,
                         handlelength=5.7, handletextpad=0.8)

# Horizon legend
horizon_handles = [
    Line2D([0], [0], color=colors[idx], linestyle='-', linewidth=2, label=rf'{h*0.006} $\Lambda_{{max}} t$') 
    for idx, h in enumerate(horizons)
]
leg_horizon = ax.legend(handles=horizon_handles, title='Prediction Horizon (h)', 
                        bbox_to_anchor=(1.01, 1), loc='upper left', frameon=True,
                        handlelength=2.5, handletextpad=0.8)

ax.add_artist(leg_features)

plt.show()

# Figure 3
---

## Individual plots

In [ ]:
expID = 0
h = 0

fig, ax = plt.subplots(figsize=(150 * mm, 100 * mm), layout='constrained')

residuals_lin = results[10000]['y_true'][expID, :, h-1] - results[10000]['y_lin'][expID, :, h-1]
residuals_nonlin = results[10000]['y_true'][expID, :, h-1] - results[10000]['y_nonlin'][expID, :, h-1]

plt.hist(residuals_lin, label='Linear', color=okabe_ito[0], edgecolor=okabe_ito[-1], alpha=.5, density=True)
plt.hist(residuals_nonlin, label='Nonlinear', color=okabe_ito[2], edgecolor=okabe_ito[-1], alpha=.5, density=True)

# Gaussian fit
mu_lin, std_lin = norm.fit(residuals_lin)
mu_nonlin, std_nonlin = norm.fit(residuals_nonlin)
x_min = min(residuals_lin.min(), residuals_nonlin.min())
x_max = max(residuals_lin.max(), residuals_nonlin.max())
x = np.linspace(x_min, x_max, 500)

ax.plot(x, norm.pdf(x, mu_lin, std_lin), label=rf'Linear fit ($\mu={mu_lin:.3f}, \sigma={std_lin:.3f}$)', color=okabe_ito[0], linestyle='--')
ax.plot(x, norm.pdf(x, mu_nonlin, std_nonlin), label=rf'Nonlinear fit ($\mu={mu_nonlin:.3f}, \sigma={std_nonlin:.3f}$)', color=okabe_ito[2], linestyle='--')

ax.spines[['top', 'right']].set_visible(False)
ax.minorticks_on()
ax.grid(which='major', linestyle='-', alpha=0.2)
ax.grid(which='minor', linestyle='--', alpha=0.1)

ax.set_title(f'Distribution of residuals at horizon = {h} (Exp. #{expID})')
ax.set_xlabel(r'Residual ($\hat{y}_h - y_h$)')
ax.legend(fontsize=8)

plt.show()

In [ ]:
def aggregate_residuals(results, amt_data, h_crop=0, n_bins=50, plot=True):
    # aggregate: [Exp, Sample, Horiz] -> [Horiz, Exp * Sample]
    y_true_all = results[amt_data]['y_true'].transpose(2, 0, 1).reshape(results[amt_data]['y_true'].shape[2], -1)
    y_lin_all = results[amt_data]['y_lin'].transpose(2, 0, 1).reshape(results[amt_data]['y_lin'].shape[2], -1)
    y_nonlin_all = results[amt_data]['y_nonlin'].transpose(2, 0, 1).reshape(results[amt_data]['y_nonlin'].shape[2], -1)

    n_horizons = y_true_all.shape[0]
    res_lin = y_true_all - y_lin_all
    res_nonlin = y_true_all - y_nonlin_all

    if plot:
        v_min, v_max = -0.4, 0.4
        bins = np.linspace(v_min, v_max, n_bins)
        
        img_lin = np.array([np.histogram(res_lin[i], bins=bins, density=True)[0] for i in range(n_horizons)])
        img_nonlin = np.array([np.histogram(res_nonlin[i], bins=bins, density=True)[0] for i in range(n_horizons)])

        fig, axs = plt.subplots(2, 2, figsize=(180 * mm, 140 * mm), 
                                gridspec_kw={'height_ratios': [1.1, 1]}, 
                                layout='constrained', sharex='col')
        
        extent = [bins[0], bins[-1], n_horizons*0.006 , 0]
        vmax_global = max(img_lin.max(), img_nonlin.max())
        p_norm = PowerNorm(gamma=0.5, vmin=0, vmax=vmax_global)
        
        # top plot
        im1 = axs[0, 0].imshow(img_lin, aspect='auto', extent=extent, cmap='afmhot', norm=p_norm)
        axs[0, 0].set_title(r'Optical RC')
        axs[0, 0].set_ylabel(r'Horizon ($\Lambda_{\max} t$)')
        
        im2 = axs[0, 1].imshow(img_nonlin, aspect='auto', extent=extent, cmap='afmhot', norm=p_norm)
        axs[0, 1].set_title(r'Optical RC + Struct. Nonlin.')

        cb = fig.colorbar(im2, ax=axs[0, :], location='right', pad=0.02, aspect=15)
        cb.set_label(r'Probability density')

        axs[1,1].sharey(axs[1,0])
        for i, (data, label, color) in enumerate(zip([res_lin, res_nonlin], ['Linear', 'Nonlinear'], [okabe_ito[0], okabe_ito[2]])):
            ax = axs[1, i]
            h_data = data[h_crop]
            mu, std = norm.fit(h_data)
            x_min, x_max = h_data.min(), h_data.max()
            x = np.linspace(x_min, x_max, len(h_data))
            
            ax.hist(h_data, bins=bins, density=True, alpha=0.9, color=color, edgecolor='black', )
            ax.plot(x, norm.pdf(x, mu, std), label=rf'$\mu={mu:.2e}$'+'\n'+rf'$\sigma={std:.2e}$', color=okabe_ito[-1], linestyle='--')
            
            ax.axvline(0, color='black', linestyle=':', alpha=0.5)
            ax.set_xlabel(r'Residual ($\epsilon = \hat{y}_h - y_h$)')
            ax.legend(fontsize=7, loc='upper right')

        for ax in axs.flat:
            ax.spines[['top', 'right']].set_visible(False)
            ax.grid(True, linestyle='--', alpha=0.3)
            ax.set_xlim(bins[0], bins[-1])
        
        axs[1, 0].set_ylabel(rf'Prob. density at $h = {h_crop*0.006} \Lambda_{{\max}} t$')
        axs[0, 0].grid(False)
        axs[0, 1].grid(False)

        for ax, color in zip(axs[0,:], [okabe_ito[0], okabe_ito[2]]):
            ax.axhline(h_crop*0.006, color=color, linestyle='-.', linewidth=.5)

        plt.show()

    return res_lin.T, res_nonlin.T

res_lin, res_nonlin = aggregate_residuals(results, amt_data=10000, h_crop=150, plot=True)

## Complete figure

In [ ]:
def plot_full_analysis(results, amt_data=10000, h_crop=150, n_bins=50, expID=0, sample_idx=1):
    y_true_all = results[amt_data]['y_true'].transpose(2, 0, 1).reshape(results[amt_data]['y_true'].shape[2], -1)
    y_lin_all = results[amt_data]['y_lin'].transpose(2, 0, 1).reshape(results[amt_data]['y_lin'].shape[2], -1)
    y_nonlin_all = results[amt_data]['y_nonlin'].transpose(2, 0, 1).reshape(results[amt_data]['y_nonlin'].shape[2], -1)

    n_horizons = y_true_all.shape[0]
    res_lin = y_true_all - y_lin_all
    res_nonlin = y_true_all - y_nonlin_all

    v_min, v_max = -0.4, 0.4 
    bins = np.linspace(v_min, v_max, n_bins)
    
    img_lin = np.array([np.histogram(res_lin[i], bins=bins, density=True)[0] for i in range(n_horizons)])
    img_nonlin = np.array([np.histogram(res_nonlin[i], bins=bins, density=True)[0] for i in range(n_horizons)])

    y_true_sample = np.hstack((results[amt_data]['y_true'][expID, sample_idx-1, :1], results[amt_data]['y_true'][expID, sample_idx, :]))
    y_lin_sample = np.hstack((results[amt_data]['y_true'][expID, sample_idx-1, :1], results[amt_data]['y_lin'][expID, sample_idx, :]))
    y_nonlin_sample = np.hstack((results[amt_data]['y_true'][expID, sample_idx-1, :1], results[amt_data]['y_nonlin'][expID, sample_idx, :]))
    tvec = np.arange(y_true_sample.shape[0]) * 0.006

    fig = plt.figure(figsize=(150 * mm, 175 * mm), layout='constrained')
    
    gs_main = fig.add_gridspec(2, 1, height_ratios=[1.8, 1])
    
    gs_left = gs_main[0].subgridspec(2, 2, height_ratios=[1, 1])
    axs = np.empty((2, 2), dtype=object)
    axs[0, 0] = fig.add_subplot(gs_left[0, 0])
    axs[0, 1] = fig.add_subplot(gs_left[0, 1], sharey=axs[0, 0])
    axs[1, 0] = fig.add_subplot(gs_left[1, 0], sharex=axs[0, 0])
    axs[1, 1] = fig.add_subplot(gs_left[1, 1], sharex=axs[0, 1], sharey=axs[1, 0])

    ax_pred = fig.add_subplot(gs_main[1, 0])

    extent = [bins[0], bins[-1], n_horizons * 0.006, 0]
    vmax_global = max(img_lin.max(), img_nonlin.max())
    p_norm = PowerNorm(gamma=0.5, vmin=0, vmax=vmax_global)
    
    im1 = axs[0, 0].imshow(img_lin, aspect='auto', extent=extent, cmap='afmhot', norm=p_norm)
    axs[0, 0].set_title(r'Linear features', fontsize=14)
    axs[0, 0].set_ylabel(r'Horizon ($\Lambda_{\max} t$)', fontsize=13, labelpad=5)
    
    im2 = axs[0, 1].imshow(img_nonlin, aspect='auto', extent=extent, cmap='afmhot', norm=p_norm)
    axs[0, 1].set_title(r'Nonlinear features', fontsize=14)

    cb = fig.colorbar(im2, ax=axs[0, 1], location='right', pad=0.02, aspect=15)
    cb.set_label(r'Probability density $\rho$', fontsize=11, labelpad=5)

    for i, (data, color) in enumerate(zip([res_lin, res_nonlin], [okabe_ito[0], okabe_ito[2]])):
        ax = axs[1, i]
        h_data = data[h_crop]
        mu, std = norm.fit(h_data)
        x = np.linspace(bins[0], bins[-1], 200)
        
        ax.hist(h_data, bins=bins, density=True, alpha=0.9, color=color, edgecolor='black')
        ax.plot(x, norm.pdf(x, mu, std), label=rf'$\mu={mu:.4f}$'+'\n'+rf'$\sigma={std:.4f}$', color=okabe_ito[-1], linestyle='--')
        
        ax.axvline(0, color='black', linestyle=':', alpha=0.5)
        ax.set_xlabel(r'Residual ($\epsilon = \hat{y}_h - y_h$)', fontsize=13, labelpad=5)
        ax.legend(loc='upper right', fontsize=7)

    for ax in axs.flat:
        ax.spines[['top', 'right']].set_visible(False)
        ax.grid(True, linestyle='--', alpha=0.3)
        ax.set_xlim(bins[0], bins[-1])
        ax.tick_params(axis='y', labelsize=11)
        ax.tick_params(axis='x', labelsize=11)
    
    axs[1, 0].set_ylabel(rf'$\rho$ at $h = {h_crop*0.006} \Lambda_{{\max}} t$', fontsize=13, labelpad=5)
    axs[0, 0].grid(False)
    axs[0, 1].grid(False)
    axs[0, 0].tick_params(labelbottom=False)
    axs[0, 1].tick_params(labelbottom=False)
    axs[1, 0].set_ylim([0, 6.5])

    for ax, color in zip(axs[0,:], [okabe_ito[0], okabe_ito[2]]):
        ax.axhline(h_crop * 0.006, color=color, linestyle='-.', linewidth=.5)


    ax_pred.plot(tvec, y_true_sample, color=okabe_ito[-1], alpha=.8, label='Ground truth', linestyle='--')

    # shaded area
    std_lin_pad = np.hstack((0, res_lin.std(axis=1)))
    std_nonlin_pad = np.hstack((0, res_nonlin.std(axis=1)))

    ax_pred.plot(tvec, y_lin_sample, color=okabe_ito[0], alpha=.8, label='Optical linear features')
    ax_pred.fill_between(tvec, y_lin_sample - std_lin_pad, y_lin_sample + std_lin_pad, color=okabe_ito[0], alpha=.2)

    ax_pred.plot(tvec, y_nonlin_sample, color=okabe_ito[2], alpha=.8, label='Optical nonlinear features')
    ax_pred.fill_between(tvec, y_nonlin_sample - std_nonlin_pad, y_nonlin_sample + std_nonlin_pad, color=okabe_ito[2], alpha=.2)

    ax_pred.spines[['top', 'right']].set_visible(False)
    ax_pred.minorticks_on()
    ax_pred.grid(which='major', linestyle='-', alpha=0.2)
    ax_pred.grid(which='minor', linestyle='--', alpha=0.1)

    ax_pred.set_ylabel(r"Predictions - $\hat{y}(t)$", fontsize=13, labelpad=5)
    ax_pred.set_xlabel(r"Prediction horizon ($\Lambda_{\max} t$)", fontsize=13, labelpad=5)
    ax_pred.legend(loc='lower right', fontsize=8)
    
    ax_pred.tick_params(axis='y', labelsize=11)
    ax_pred.tick_params(axis='x', labelsize=11)

    plt.show()

plot_full_analysis(results, amt_data=10000, h_crop=150, n_bins=50, expID=0, sample_idx=7)

# Figure 4
---

## NMSE across experiments

In [ ]:
# Optical reservoir computing
inst_nmse_lin, inst_nmse_nonlin, run_nmse_lin, run_nmse_nonlin = calculate_full_nmse(results=results, amt_data=10000, nb_samples=5000, samples_spacing=1)

run_nmse_lin, run_nmse_nonlin = run_nmse_lin.mean(axis=1), run_nmse_nonlin.mean(axis=1) # (N_exp, h)
# run_nmse_lin, run_nmse_nonlin = run_nmse_lin.reshape(-1, 500), run_nmse_nonlin.reshape(-1, 500) # (N_exp*N_samples, h) -> equivalent, but the bootstrap has more samples, so the confidence interval is very narrow around the mean

# Bootstrap
_, _, _, low_boot_pct_lin, high_boot_pct_lin = bootstrap(run_nmse_lin, n_boot=500)
_, _, _, low_boot_pct_nonlin, high_boot_pct_nonlin = bootstrap(run_nmse_nonlin, n_boot=500)

tvec = np.arange(1, 501) * 0.006

fig, ax = plt.subplots(
        figsize=(200 * mm, 75 * mm),
        layout='constrained',
    )

ax.plot(np.hstack([0, tvec]), np.hstack([0, run_nmse_lin.mean(axis=0)]), color=okabe_ito[0], label='Optical RC', linewidth=2)
ax.plot(np.hstack([0, tvec]), np.hstack([0, run_nmse_nonlin.mean(axis=0)]), color=okabe_ito[2], label='Optical RC + Structural Nonlinearity', linewidth=2)

# ax.plot(tvec, boot_mean_lin, color=okabe_ito[-1], label='Boot Linear', linestyle='--', linewidth=1)
ax.fill_between(tvec, low_boot_pct_lin, high_boot_pct_lin, color=okabe_ito[0], alpha=.2)
# ax.plot(tvec, boot_mean_nonlin, color=okabe_ito[-1], label='Boot Nonlinear', linestyle='--', linewidth=1)
ax.fill_between(tvec, low_boot_pct_nonlin, high_boot_pct_nonlin, color=okabe_ito[2], alpha=.2)

# ax.set_title(f'Average Reservoir Performances')
ax.set_xlabel(r"Prediction horizon ($\Lambda_{\max} t$)", fontsize=14, labelpad=5)
ax.set_ylabel(r'NMSE', fontsize=14, labelpad=5)
ax.spines[['top', 'right']].set_visible(False)
ax.minorticks_on()
ax.grid(which='major', linestyle='-', alpha=0.2)
ax.grid(which='minor', linestyle='--', alpha=0.1)
ax.tick_params(axis='x', labelsize=12)
ax.tick_params(axis='y', labelsize=12)
# ax.set_ylim([0.0, 0.3])
ax.legend(fontsize=12, loc='lower right')
# plt.tight_layout()
plt.show()

## Diebold Mariano

In [ ]:
def blend_with(color, target=(1, 1, 1), alpha=0.5):
    """
    Mistura uma cor com outra.
    alpha=0   -> cor original
    alpha=1   -> target
    """
    c = np.array(mcolors.to_rgb(color))
    t = np.array(target)
    return tuple((1 - alpha) * c + alpha * t)

def make_segmented_log_cmap(
    bounds,
    base_colors,
    total_samples=1024,
    dark_alpha=0.45,
    light_alpha=0.35,
):
    """
    Cria uma colormap compatível com LogNorm.

    Parameters
    ----------
    bounds : list
        Limites das faixas em escala real.
        Ex:
            [1e-20, 1e-13, 1e-7, 1e-3, 0.05, 1.0]

    base_colors : list
        Uma cor-base por faixa.
        len(base_colors) == len(bounds)-1

    total_samples : int
        Número total de cores da colormap.

    dark_alpha : float
        Escurecimento no início de cada faixa.

    light_alpha : float
        Clareamento no final de cada faixa.
    """

    assert len(base_colors) == len(bounds) - 1

    log_bounds = np.log10(bounds)

    # largura de cada faixa em décadas
    widths = np.diff(log_bounds)

    # distribui resolução proporcionalmente à largura log
    n_per_segment = np.maximum(
        2,
        np.round(total_samples * widths / widths.sum()).astype(int)
    )

    all_colors = []

    for base, n in zip(base_colors, n_per_segment):

        c_dark = blend_with(
            base,
            target=(0, 0, 0),
            alpha=dark_alpha
        )

        c_light = blend_with(
            base,
            target=(1, 1, 1),
            alpha=light_alpha
        )

        t = np.linspace(0, 1, n)

        segment_colors = [
            tuple(
                (1 - x) * np.array(c_dark)
                + x * np.array(c_light)
            )
            for x in t
        ]

        all_colors.extend(segment_colors)

    return LinearSegmentedColormap.from_list(
        "segmented_log_cmap",
        all_colors,
        N=len(all_colors)
    )

bounds = [1e-20, 1e-13, 1e-7, 1e-3, 0.05, 1.0]
base_colors = [
    "#2C1E7F",  # deep indigo (extremo)
    "#1F5AA6",  # azul forte
    "#1FA187",  # teal (transição)
    "#7AD151",  # verde claro
    "#FDE725",  # amarelo (não significativo)
    # "#FD2525"
]

custom_cmap = make_segmented_log_cmap(
    bounds,
    base_colors,
    total_samples=2048,
    dark_alpha=0.7,
    light_alpha=0.2
)

custom_norm = LogNorm(vmin=bounds[0], vmax=bounds[-1])

In [ ]:
def plot_diebold_mariano(results, amt_data):
    tvec = np.arange(1, 501) * 0.006

    y_lin = results[amt_data]['y_lin'][:, :, :500]
    y_true = results[amt_data]['y_true'][:, :, :500]
    y_nonlin = results[amt_data]['y_nonlin'][:, :, :500]

    dm_test_experiments = []

    for expID in range(y_lin.shape[0]):
        p_vals = []
        for h in tqdm(range(0, 500), desc=f'ExpID: {expID}'):
            d = (y_nonlin[expID, :, h] - y_true[expID, :, h])**2 - (y_lin[expID, :, h] - y_true[expID, :, h])**2
            stat = dm_test(d=d, h=h+1, alpha=0.05)
            p_vals.append(2 * scp.stats.norm.sf(np.abs(stat['dm'])))
            
        dm_test_experiments.append(np.array(p_vals))

    dm_test_experiments = np.array(dm_test_experiments)

    fig, ax = plt.subplots(
            figsize=(120 * mm, 100 * mm),
            layout='constrained',
        )

    dm_plot = np.ma.masked_less_equal(dm_test_experiments, 0)

    im = ax.imshow(
        dm_plot,
        aspect='auto',
        origin='lower',
        cmap=custom_cmap,
        norm=custom_norm,
        extent=[0.0, tvec[-1], 0, dm_plot.shape[0]],
        interpolation='nearest'
    )

    ax.set_title("Diebold-Mariano test")
    ax.set_ylabel("Experiment Nb.")
    ax.set_xlabel(r"Prediction horizon ($\Lambda_{\max} t$)")
    ax.grid(False)

    cbar = fig.colorbar(im, ax=ax)
    cbar.set_label(r"$p$-value")
    plt.show()

plot_diebold_mariano(results, amt_data=10000)

## Spectral distance

In [ ]:
spectral_dist_results = spectral_distance(
    results,
    amt_data=10000,
    nb_samples=5000,
    samples_spacing=1,
    dt=1,
    window_size=500
)

In [ ]:
data = [spectral_dist_results['spectral_distance_lin'].mean(axis=1),
        spectral_dist_results['spectral_distance_nonlin'].mean(axis=1)]


fig, ax = plt.subplots(figsize=(6, 5))

bp = ax.boxplot(
    data, 
    widths=0.5, 
    showfliers=False,
    patch_artist=True,
    zorder=0
)

colors = [okabe_ito[0], okabe_ito[2]]
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(mcolors.to_rgba(color, alpha=0.2))
    patch.set_edgecolor(color)
    patch.set_linewidth(1.5)

for element in ['whiskers', 'caps']:
    plt.setp(bp[element], color='#2c3e50', linewidth=1.5)

for median in bp['medians']:
    plt.setp(median, color='black', linewidth=1.5)

for i, y in enumerate(data, start=1):

    x = np.ones(len(y))*i

    ax.scatter(
        x,
        y,
        alpha=0.5,
        s=20,
        color=okabe_ito[0] if i == 1 else okabe_ito[2]
    )

# Labels
ax.set_xticks([1, 2])
ax.set_xticklabels(['Linear', 'Nonlinear'], fontdict={'fontsize': 14})

ax.set_ylabel('Spectral Distance', fontsize=14, labelpad=5)

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_linewidth(0.8)
ax.spines['bottom'].set_linewidth(0.8)
ax.grid(axis='y', linestyle='--', alpha=0.5, zorder=0)
ax.tick_params(axis='y', labelsize=12)

plt.tight_layout()
plt.show()

## L1 distance

In [ ]:
def compute_l1(results, amt_data=10000):
    nbins = int(np.sqrt(500))
    n_exp = results[amt_data]['y_true'].shape[0]

    pos_l1_dist_lin = np.zeros((n_exp, 5000))
    pos_l1_dist_nonlin = np.zeros((n_exp, 5000))

    vel_l1_dist_lin = np.zeros((n_exp, 5000))
    vel_l1_dist_nonlin = np.zeros((n_exp, 5000))

    for exp_idx in range(n_exp):

        all_pos = np.concatenate(
            [
                results[amt_data]['y_true'][exp_idx, :, :].ravel(),
                results[amt_data]['y_lin'][exp_idx, :, :].ravel(),
                results[amt_data]['y_nonlin'][exp_idx, :, :].ravel()
            ]
        )
        _, pos_edges = np.histogram(all_pos, bins=nbins)

        all_vel = np.concatenate(
            [
                np.diff(results[amt_data]['y_true'][exp_idx, :, :]).ravel(),
                np.diff(results[amt_data]['y_lin'][exp_idx, :, :]).ravel(),
                np.diff(results[amt_data]['y_nonlin'][exp_idx, :, :]).ravel()
            ]
        )
        _, vel_edges = np.histogram(all_vel, bins=nbins)
        

        for sample_idx in range(5000):

            # Position histograms
            counts_true, _ = np.histogram(results[amt_data]['y_true'][exp_idx, sample_idx, :], bins=pos_edges)
            counts_lin, _ = np.histogram(results[amt_data]['y_lin'][exp_idx, sample_idx, :], bins=pos_edges)
            counts_nonlin, _ = np.histogram(results[amt_data]['y_nonlin'][exp_idx, sample_idx, :], bins=pos_edges)

            p_pos_true = counts_true / (counts_true.sum() + 1e-12)
            p_pos_lin = counts_lin / (counts_lin.sum() + 1e-12)
            p_pos_nonlin = counts_nonlin / (counts_nonlin.sum() + 1e-12)

            pos_l1_dist_lin[exp_idx, sample_idx] = np.sum(np.abs(p_pos_lin - p_pos_true))
            pos_l1_dist_nonlin[exp_idx, sample_idx] = np.sum(np.abs(p_pos_nonlin - p_pos_true))

            # Velocity histograms
            counts_true, _ = np.histogram(np.diff(results[amt_data]['y_true'][exp_idx, sample_idx, :]), bins=vel_edges)
            counts_lin, _ = np.histogram(np.diff(results[amt_data]['y_lin'][exp_idx, sample_idx, :]), bins=vel_edges)
            counts_nonlin, _ = np.histogram(np.diff(results[amt_data]['y_nonlin'][exp_idx, sample_idx, :]), bins=vel_edges)

            p_vel_true = counts_true / (counts_true.sum() + 1e-12)
            p_vel_lin = counts_lin / (counts_lin.sum() + 1e-12)
            p_vel_nonlin = counts_nonlin / (counts_nonlin.sum() + 1e-12)

            vel_l1_dist_lin[exp_idx, sample_idx] = np.sum(np.abs(p_vel_lin - p_vel_true))
            vel_l1_dist_nonlin[exp_idx, sample_idx] = np.sum(np.abs(p_vel_nonlin - p_vel_true))
        
    return pos_l1_dist_lin, vel_l1_dist_lin, pos_l1_dist_nonlin, vel_l1_dist_nonlin

pos_l1_dist_lin, vel_l1_dist_lin, pos_l1_dist_nonlin, vel_l1_dist_nonlin = compute_l1(results, amt_data=10000)

In [ ]:
data = [(pos_l1_dist_lin.mean(axis=1) + vel_l1_dist_lin.mean(axis=1)) / 2, (pos_l1_dist_nonlin.mean(axis=1) + vel_l1_dist_nonlin.mean(axis=1)) / 2]


fig, ax = plt.subplots(figsize=(6, 5))

bp = ax.boxplot(
    data, 
    widths=0.5, 
    showfliers=False,
    patch_artist=True,
    zorder=0
)

colors = [okabe_ito[0], okabe_ito[2]]
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(mcolors.to_rgba(color, alpha=0.2))
    patch.set_edgecolor(color)
    patch.set_linewidth(1.5)

colors = [okabe_ito[0], okabe_ito[2]]
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(mcolors.to_rgba(color, alpha=0.2))
    patch.set_edgecolor(color)
    patch.set_linewidth(1.5)

for element in ['whiskers', 'caps']:
    plt.setp(bp[element], color='#2c3e50', linewidth=1.5)

for median in bp['medians']:
    plt.setp(median, color='black', linewidth=1.5)

for i, y in enumerate(data, start=1):

    x = np.ones(len(y))*i

    ax.scatter(
        x,
        y,
        alpha=0.5,
        s=20,
        color=okabe_ito[0] if i == 1 else okabe_ito[2]
    )


# Labels
ax.set_xticks([1, 2])
ax.set_xticklabels(['Linear', 'Nonlinear'], fontdict={'fontsize': 14})

ax.set_ylabel(r'$L_1$ error', fontsize=14, labelpad=5)

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_linewidth(0.8)
ax.spines['bottom'].set_linewidth(0.8)
ax.grid(axis='y', linestyle='--', alpha=0.5, zorder=0)
ax.tick_params(axis='y', labelsize=12)

plt.tight_layout()
plt.show()

## Complete figure (Spectral Distance, L1, NMSE)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.gridspec as gridspec

fig = plt.figure(figsize=(200 * mm, 150 * mm), layout='constrained')


gs = gridspec.GridSpec(2, 2, figure=fig, height_ratios=[1, 0.7])

ax0 = fig.add_subplot(gs[1, :])
ax1 = fig.add_subplot(gs[0, 0])
ax2 = fig.add_subplot(gs[0, 1])


# NMSE plot
ax0.plot(np.hstack([0, tvec]), np.hstack([0, run_nmse_lin.mean(axis=0)]), color=okabe_ito[0], label='Optical RC', linewidth=2)
ax0.plot(np.hstack([0, tvec]), np.hstack([0, run_nmse_nonlin.mean(axis=0)]), color=okabe_ito[2], label='Optical RC + Struct. Nonlin.', linewidth=2)

ax0.fill_between(tvec, low_boot_pct_lin, high_boot_pct_lin, color=okabe_ito[0], alpha=0.2)
ax0.fill_between(tvec, low_boot_pct_nonlin, high_boot_pct_nonlin, color=okabe_ito[2], alpha=0.2)

ax0.set_xlabel(r"Prediction horizon ($\Lambda_{\max} t$)", fontsize=13, labelpad=5)
ax0.set_ylabel(r'NMSE', fontsize=13, labelpad=5)
ax0.spines[['top', 'right']].set_visible(False)
ax0.minorticks_on()
ax0.grid(which='major', linestyle='-', alpha=0.15)
ax0.grid(which='minor', linestyle='--', alpha=0.08)
ax0.tick_params(axis='both', labelsize=11)
ax0.legend(fontsize=11, loc='lower right')

# Spectral distance
data_sd = [spectral_dist_results['spectral_distance_lin'].mean(axis=1),
           spectral_dist_results['spectral_distance_nonlin'].mean(axis=1)]

bp1 = ax1.boxplot(data_sd, widths=0.45, showfliers=False, patch_artist=True, zorder=2)

colors = [okabe_ito[0], okabe_ito[2]]
for patch, color in zip(bp1['boxes'], colors):
    patch.set_facecolor(mcolors.to_rgba(color, alpha=0.2))
    patch.set_edgecolor(color)
    patch.set_linewidth(1.5)

for element in ['whiskers', 'caps']:
    plt.setp(bp1[element], color='#2c3e50', linewidth=1.2)
for median in bp1['medians']:
    plt.setp(median, color='black', linewidth=1.5)

for i, y in enumerate(data_sd, start=1):
    x = np.random.normal(i, 0.00, size=len(y))
    ax1.scatter(x, y, alpha=0.6, s=15, color=colors[i-1], zorder=3)

ax1.set_xticks([1, 2])
ax1.set_xticklabels(['Linear', 'Nonlinear'], fontsize=12)
ax1.set_ylabel('Spectral Distance', fontsize=13, labelpad=5)
ax1.spines[['top', 'right']].set_visible(False)
ax1.grid(axis='y', linestyle='--', alpha=0.4, zorder=1)
ax1.tick_params(axis='y', labelsize=11)

# L1 histogram error
data_l1 = [(pos_l1_dist_lin.mean(axis=1) + vel_l1_dist_lin.mean(axis=1)) / 2, 
           (pos_l1_dist_nonlin.mean(axis=1) + vel_l1_dist_nonlin.mean(axis=1)) / 2]

bp2 = ax2.boxplot(data_l1, widths=0.45, showfliers=False, patch_artist=True, zorder=2)

for patch, color in zip(bp2['boxes'], colors):
    patch.set_facecolor(mcolors.to_rgba(color, alpha=0.2))
    patch.set_edgecolor(color)
    patch.set_linewidth(1.5)

for element in ['whiskers', 'caps']:
    plt.setp(bp2[element], color='#2c3e50', linewidth=1.2)
for median in bp2['medians']:
    plt.setp(median, color='black', linewidth=1.5)

for i, y in enumerate(data_l1, start=1):
    x = np.random.normal(i, 0.00, size=len(y))
    ax2.scatter(x, y, alpha=0.6, s=15, color=colors[i-1], zorder=3)

ax2.set_xticks([1, 2])
ax2.set_xticklabels(['Linear', 'Nonlinear'], fontsize=12)
ax2.set_ylabel(r'$L_1$ error', fontsize=13, labelpad=5)
ax2.spines[['top', 'right']].set_visible(False)
ax2.grid(axis='y', linestyle='--', alpha=0.4, zorder=1)
ax2.tick_params(axis='y', labelsize=11)

plt.show()